# 03 — Regression Analysis: OD vs SD (statsmodels OLS, Parallelised)

## Objective
For every matched pair of **Original Data (OD)** and **Synthetic Data (SD)**, fit an OLS
regression using `statsmodels` (not `sklearn`) so that we obtain:

| Metric | Why we need it |
|--------|----------------|
| $\hat{\beta}$ coefficients | Euclidean distance, coefficient bias |
| Standard errors (SE) | Confidence-interval overlap |
| $p$-values | Significance agreement |
| Adjusted $R^2$ | $R^2$ degradation |

### Design decisions
1. **No `.pkl` model files.** We fit → extract metrics → discard the model object.
2. **`joblib.Parallel`** distributes work across all available CPU cores.
3. **Single Parquet output** (`results/aggregated_model_metrics.parquet`) for fast downstream I/O.
4. **`statsmodels.api.OLS`** gives standard errors and $p$-values natively,
   unlike `sklearn.linear_model.LinearRegression`.

### Simulation grid
- 3 synthesis methods: CART, NORM, PMM
- 2 variable types: continuous, binary
- 2 sample sizes: $N \in \{100, 1000\}$
- 2 dimensionalities: $p \in \{3, 10\}$
- 3 correlation levels: $\rho \in \{0.0, 0.3, 0.6\}$
- 3 error variances: $\sigma^2 \in \{1.0, 1.5, 2.0\}$
- $M$ repetitions per scenario

In [14]:
# ── Imports ──────────────────────────────────────────────────────────────────
import json, os, re, glob, gc, warnings, time
from multiprocessing.shared_memory import SharedMemory

import numpy as np
import pandas as pd
import statsmodels.api as sm
from joblib import Parallel, delayed

warnings.filterwarnings("ignore", category=FutureWarning)
print(f"NumPy {np.__version__}  •  pandas {pd.__version__}  •  statsmodels {sm.__version__}")


NumPy 2.4.2  •  pandas 3.0.1  •  statsmodels 0.14.6


In [2]:
# ── Load Configuration ──────────────────────────────────────────────────────
config_path = os.path.join("..", "config", "config.json")
with open(config_path) as f:
    config = json.load(f)

M            = config["simulation"]["M"]
N_list       = config["simulation"]["N"]
p_list       = config["simulation"]["p"]
var_types    = config["simulation"]["var_type"]
rho_vals     = config["parameters"]["rho"]
sigma_2_vals = config["parameters"]["sigma_2"]
beta_full    = np.array(config["parameters"]["beta"])
methods      = config["synthesis"]["methods"]

print(json.dumps(config, indent=2))
print(f"\nGrid: {len(methods)} methods × {len(var_types)} var_types"
      f" × {len(N_list)} N × {len(p_list)} p"
      f" × {len(rho_vals)} ρ × {len(sigma_2_vals)} σ² × M={M} reps")
print(f"Methods: {methods}")

{
  "simulation": {
    "N": [
      100,
      1000
    ],
    "p": [
      3,
      10
    ],
    "var_type": [
      "continuous",
      "binary"
    ],
    "random_seed_base": 16,
    "test_seed": 123,
    "N_test": 10000,
    "M": 10
  },
  "parameters": {
    "rho": [
      0.0,
      0.3,
      0.6
    ],
    "sigma_2": [
      1.0,
      1.5,
      2.0
    ],
    "p1": [
      0.5
    ],
    "beta": [
      1,
      1,
      1,
      1,
      1,
      1,
      1,
      1,
      1,
      1,
      1
    ]
  },
  "synthesis": {
    "methods": [
      "cart",
      "norm",
      "pmm"
    ]
  }
}

Grid: 3 methods × 2 var_types × 2 N × 2 p × 3 ρ × 3 σ² × M=10 reps
Methods: ['cart', 'norm', 'pmm']


In [3]:
import glob, re
import pyarrow as pa
import pyarrow.parquet as pq

# ── Paths ─────────────────────────────────────────────────────────────────────
od_dir         = os.path.join("..", "data", "original")
sd_dir         = os.path.join("..", "data", "synthetic")
result_dir     = os.path.join("..", "results")
od_packed_path = os.path.join("..", "data", "OD_packed.parquet")
sd_packed_path = os.path.join("..", "data", "SD_packed.parquet")
os.makedirs(result_dir, exist_ok=True)

# ── Filename-parsing helpers ───────────────────────────────────────────────────
_OD_RE = re.compile(
    r"OD_(binary|continuous)_N(\d+)_p(\d+)_rho([\d.]+)_sig([\d.]+)_p1[\w.]+_iter(\d+)"
)
_SD_RE = re.compile(
    r"SD_(\w+?)_(binary|continuous)_N(\d+)_p(\d+)_rho([\d.]+)_sig([\d.]+)_p1[\w.]+_iter(\d+)"
)

# Maximum dimensionality in the simulation grid — fixes the column schema
MAX_P    = 10
FEAT_COLS = [f"X{i}" for i in range(1, MAX_P + 1)]   # X1 … X10
SCHEMA_COLS = FEAT_COLS + ["y"]


def _read_one_od(fp):
    """Read one OD CSV and return a DataFrame with metadata columns appended."""
    m = _OD_RE.search(os.path.basename(fp))
    vtype, N, p, rho, sig, it = (m.group(1), int(m.group(2)),
                                  int(m.group(3)), float(m.group(4)),
                                  float(m.group(5)), int(m.group(6)))
    df = pd.read_csv(fp, dtype="float32")
    df.columns = [c.strip('"') for c in df.columns]       # strip R-style quotes
    # Pad missing feature columns with NaN (for p<MAX_P files)
    for c in SCHEMA_COLS:
        if c not in df.columns:
            df[c] = float("nan")
    df = df[SCHEMA_COLS].copy()
    df["var_type"] = vtype
    df["N"]        = np.int16(N)
    df["p"]        = np.int8(p)
    df["rho"]      = np.float32(rho)
    df["sigma_2"]  = np.float32(sig)
    df["iter"]     = np.int16(it)
    return df


def _read_one_sd(fp):
    """Read one SD CSV and return a DataFrame with metadata columns appended."""
    m = _SD_RE.search(os.path.basename(fp))
    method, vtype, N, p, rho, sig, it = (m.group(1), m.group(2),
                                          int(m.group(3)), int(m.group(4)),
                                          float(m.group(5)), float(m.group(6)),
                                          int(m.group(7)))
    df = pd.read_csv(fp, dtype="float32")
    df.columns = [c.strip('"') for c in df.columns]
    for c in SCHEMA_COLS:
        if c not in df.columns:
            df[c] = float("nan")
    df = df[SCHEMA_COLS].copy()
    df["method"]  = method
    df["var_type"] = vtype
    df["N"]        = np.int16(N)
    df["p"]        = np.int8(p)
    df["rho"]      = np.float32(rho)
    df["sigma_2"]  = np.float32(sig)
    df["iter"]     = np.int16(it)
    return df


# ── One-time CSV → Parquet packing ────────────────────────────────────────────
# Skipped automatically on subsequent runs when both files already exist.

def _pack(csv_glob, reader_fn, out_path, label, n_jobs=-1):
    files = sorted(glob.glob(csv_glob))
    assert files, f"No files matched: {csv_glob}"
    print(f"  Packing {len(files):,} {label} CSVs → {out_path}")
    t = time.time()
    chunks = Parallel(n_jobs=n_jobs, verbose=5, backend="loky")(
        delayed(reader_fn)(fp) for fp in files
    )
    print(f"  Concatenating …")
    big = pd.concat(chunks, ignore_index=True)
    del chunks; gc.collect()
    print(f"  Writing parquet …")
    big.to_parquet(out_path, index=False, engine="pyarrow",
                   compression="snappy", row_group_size=50_000)
    del big; gc.collect()
    print(f"  ✓ Done in {(time.time()-t)/60:.1f} min  →  "
          f"{os.path.getsize(out_path)/1e9:.2f} GB")


if not os.path.exists(od_packed_path):
    print("OD_packed.parquet not found — packing now (one-time, ~few minutes) …")
    _pack(os.path.join(od_dir, "OD_*.csv"), _read_one_od, od_packed_path, "OD")
else:
    print(f"OD_packed.parquet already exists ({os.path.getsize(od_packed_path)/1e9:.2f} GB)")

if not os.path.exists(sd_packed_path):
    print("SD_packed.parquet not found — packing now (one-time, ~several minutes) …")
    _pack(os.path.join(sd_dir, "SD_*.csv"), _read_one_sd, sd_packed_path, "SD")
else:
    print(f"SD_packed.parquet already exists ({os.path.getsize(sd_packed_path)/1e9:.2f} GB)")

# ── Load into memory ───────────────────────────────────────────────────────────
print("\nLoading OD_packed.parquet …")
od_df = pq.read_table(od_packed_path).to_pandas()
print(f"  shape: {od_df.shape}   mem: {od_df.memory_usage(deep=True).sum()/1e9:.2f} GB")

print("Loading SD_packed.parquet …")
sd_df = pq.read_table(sd_packed_path).to_pandas()
print(f"  shape: {sd_df.shape}   mem: {sd_df.memory_usage(deep=True).sum()/1e9:.2f} GB")

# Categoricals save RAM for groupby operations below
for _df in [od_df, sd_df]:
    for _col in ["var_type"]:
        _df[_col] = _df[_col].astype("category")
if "method" in sd_df.columns:
    sd_df["method"] = sd_df["method"].astype("category")

print("\n✓ Both datasets in memory and ready.")


OD_packed.parquet already exists (1.18 GB)
SD_packed.parquet already exists (1.76 GB)

Loading OD_packed.parquet …
  shape: (39996000, 17)   mem: 4.04 GB
Loading SD_packed.parquet …
  shape: (119988000, 18)   mem: 13.52 GB

✓ Both datasets in memory and ready.


In [4]:
# ── Build SharedMemory arenas: zero-copy IPC for worker processes ─────────────
#
# Cleanup guard: release any SharedMemory left from a previous (failed) run
# BEFORE allocating new segments.  This makes the cell safe to re-run.
for _shm_var in ("shm_od", "shm_sd"):
    _shm = globals().get(_shm_var)
    if _shm is not None:
        try:
            _shm.close()
            _shm.unlink()
            print(f"Released stale {_shm_var} ({_shm.name})")
        except Exception as _e:
            print(f"Could not release {_shm_var}: {_e}")
        globals().pop(_shm_var, None)

# ── Constants ─────────────────────────────────────────────────────────────────
OD_META    = ["var_type", "N", "p", "rho", "sigma_2", "iter"]
SD_META    = ["method",   "var_type", "N", "p", "rho", "sigma_2", "iter"]
ARENA_COLS = MAX_P + 1   # MAX_P features (0-padded) + y


def _col_to_int_key(series):
    """Return a stable int64 sort key for any column (strings → dense codes)."""
    vals = series.to_numpy()
    if vals.dtype.kind in ("U", "O"):
        _, codes = np.unique(vals, return_inverse=True)
        return codes.astype(np.int64)
    if vals.dtype.kind in ("i", "u"):
        return vals.astype(np.int64)
    return vals.astype(np.float64)


def _build_arena(df, meta_cols, feat_cols, max_p, arena_cols, label):
    """
    Sort df by meta_cols via np.lexsort, pack into a float64 arena,
    and return (arena, index_dict).
    """
    n_rows = len(df)
    print(f"  [{label}] {n_rows:,} rows — building sort permutation …", flush=True)
    t0 = time.time()

    int_keys = [_col_to_int_key(df[c]) for c in meta_cols]
    sort_idx  = np.lexsort(int_keys[::-1])
    print(f"  [{label}] permutation built in {time.time()-t0:.1f}s", flush=True)

    t0 = time.time()
    arena = np.empty((n_rows, arena_cols), dtype=np.float64)
    arena[:, :max_p] = df[feat_cols].to_numpy(dtype=np.float64, na_value=np.nan)[sort_idx]
    arena[:, max_p]  = df["y"].to_numpy(dtype=np.float64)[sort_idx]
    print(f"  [{label}] arena packed in {time.time()-t0:.1f}s", flush=True)

    t0 = time.time()
    sorted_int_keys = [k[sort_idx] for k in int_keys]

    boundary_masks = [sorted_int_keys[i][1:] != sorted_int_keys[i][:-1]
                      for i in range(len(meta_cols))]
    boundaries = np.where(np.logical_or.reduce(boundary_masks))[0] + 1
    starts = np.concatenate([[0], boundaries])
    ends   = np.concatenate([boundaries, [n_rows]])

    orig_col_arrays = [df[c].to_numpy()[sort_idx] for c in meta_cols]

    idx_map = {}
    for s, e in zip(starts, ends):
        key = tuple(arr[s].item() if hasattr(arr[s], "item") else arr[s]
                    for arr in orig_col_arrays)
        idx_map[key] = (int(s), int(e))

    print(f"  [{label}] index built in {time.time()-t0:.1f}s  |  groups: {len(idx_map):,}", flush=True)
    print(f"  [{label}] arena {arena.shape}  ({arena.nbytes/1e9:.2f} GB)")
    return arena, idx_map


# ── Pack OD ───────────────────────────────────────────────────────────────────
print("=== Packing OD ===")
t_total = time.time()
od_arena, od_index = _build_arena(od_df, OD_META, FEAT_COLS, MAX_P, ARENA_COLS, "OD")
print(f"OD total: {time.time()-t_total:.1f}s\n")

# ── Pack SD + build work_items ─────────────────────────────────────────────────
print("=== Packing SD ===")
t_total = time.time()
sd_arena, sd_index = _build_arena(sd_df, SD_META, FEAT_COLS, MAX_P, ARENA_COLS, "SD")
print(f"SD total: {time.time()-t_total:.1f}s\n")

work_items, skipped_no_od = [], 0
for key, (sd_s, sd_e) in sd_index.items():
    method, vtype, N, p_val, rho, sigma, it = key
    od_key = (vtype, N, p_val, rho, sigma, it)
    if od_key not in od_index:
        skipped_no_od += 1
        continue
    od_s, od_e = od_index[od_key]
    work_items.append((od_s, od_e, sd_s, sd_e, int(p_val), key))

del sd_index, od_index
gc.collect()
print(f"work items: {len(work_items):,}  |  skipped (no OD): {skipped_no_od:,}\n")

# ── Allocate SharedMemory ─────────────────────────────────────────────────────
print("Allocating shared memory …", flush=True)
shm_od = SharedMemory(create=True, size=od_arena.nbytes)
shm_sd = SharedMemory(create=True, size=sd_arena.nbytes)

np.ndarray(od_arena.shape, dtype=np.float64, buffer=shm_od.buf)[:] = od_arena
np.ndarray(sd_arena.shape, dtype=np.float64, buffer=shm_sd.buf)[:] = sd_arena

OD_SHM_NAME, OD_SHAPE = shm_od.name, od_arena.shape
SD_SHM_NAME, SD_SHAPE = shm_sd.name, sd_arena.shape

del od_arena, sd_arena
gc.collect()

print(f"  OD  shm://{OD_SHM_NAME}   {OD_SHAPE}")
print(f"  SD  shm://{SD_SHM_NAME}   {SD_SHAPE}")
print("✓ Shared memory ready — workers will zero-copy slice by row range.")


=== Packing OD ===
  [OD] 39,996,000 rows — building sort permutation …
  [OD] permutation built in 13.9s
  [OD] arena packed in 1.1s
  [OD] index built in 1.7s  |  groups: 72,000
  [OD] arena (39996000, 11)  (3.52 GB)
OD total: 16.9s

=== Packing SD ===
  [SD] 119,988,000 rows — building sort permutation …
  [SD] permutation built in 85.8s
  [SD] arena packed in 22.5s
  [SD] index built in 18.5s  |  groups: 216,000
  [SD] arena (119988000, 11)  (10.56 GB)
SD total: 127.8s

work items: 216,000  |  skipped (no OD): 0

Allocating shared memory …
  OD  shm://psm_2de62863   (39996000, 11)
  SD  shm://psm_f2887e1c   (119988000, 11)
✓ Shared memory ready — workers will zero-copy slice by row range.


In [5]:
# ── Work-items sanity check ───────────────────────────────────────────────────
print(f"Total (OD, SD) pairs to process : {len(work_items):,}")

# Spot-check first entry
od_s, od_e, sd_s, sd_e, p_val, meta = work_items[0]
method_0, vtype_0, N_meta, p_meta, rho_0, sigma_0, it_0 = meta

od_rows = od_e - od_s
sd_rows = sd_e - sd_s

print(f"\nFirst work item:")
print(f"  meta           : {meta}")
print(f"  p (from meta)  : {p_meta}  |  arena p_val stored : {p_val}")
print(f"  OD rows [{od_s}, {od_e})  → {od_rows} rows  (N in meta = {N_meta})")
print(f"  SD rows [{sd_s}, {sd_e})  → {sd_rows} rows  (N in meta = {N_meta})")

if od_rows != N_meta:
    print(f"  ⚠ OD row count {od_rows} ≠ N={N_meta} — "
          f"each group may span multiple packed rows per scenario.")
if sd_rows != N_meta:
    print(f"  ⚠ SD row count {sd_rows} ≠ N={N_meta}")

# Structural check: every work_item has matching OD and SD row counts
mismatched = [(i, it) for i, it in enumerate(work_items) if (it[1]-it[0]) != (it[3]-it[2])]
if mismatched:
    print(f"\n⚠ {len(mismatched)} work items have mismatched OD/SD row counts — first 5: {mismatched[:5]}")
else:
    print(f"\n✓ All {len(work_items):,} work items have matching OD/SD row counts.")

print(f"\n{'✓' if p_val == p_meta else '⚠'} Arena p_val == meta p: {p_val == p_meta}")


Total (OD, SD) pairs to process : 216,000

First work item:
  meta           : ('cart', 'binary', 100, 3, 0.0, 1.0, 1)
  p (from meta)  : 3  |  arena p_val stored : 3
  OD rows [0, 200)  → 200 rows  (N in meta = 100)
  SD rows [0, 200)  → 200 rows  (N in meta = 100)
  ⚠ OD row count 200 ≠ N=100 — each group may span multiple packed rows per scenario.
  ⚠ SD row count 200 ≠ N=100

✓ All 216,000 work items have matching OD/SD row counts.

✓ Arena p_val == meta p: True


## Fit OLS Models (Parallelised — Zero-Copy SharedMemory Architecture)

### Bottleneck eliminated
The original architecture spawned one `joblib` worker **per CSV file** (218,160 files),
causing the Ryzen 9 to idle at ≈11% utilisation because every worker was waiting on
disk I/O and inter-process pickle serialisation.

### New data flow
```
Disk  ──►  od_df / sd_df (Parquet, loaded once)
               │
               ▼
      SharedMemory arenas  (OD_SHAPE, SD_SHAPE float64 arrays)
          /            \
   worker 0         worker N          ← attach by name, slice by row index
   no copy!         no copy!
```

1. **Parquet → RAM** in the main process once (`pd.read_parquet`).
2. **RAM → SharedMemory** once (`shm.buf` memcpy). Workers then attach by OS name — no further copying.
3. Each worker receives only a **tiny `work_items` tuple** (8 integers + metadata), not the data itself.
4. Workers slice `np.ndarray` views directly from the shared buffer. OLS runs on the view.
5. Only the small result **dict** is pickled back — identical to the original output contract.

### Statistical rationale (unchanged)
`statsmodels.api.OLS` is used (not `sklearn`) because we need **standard errors**,
**$p$-values**, and the full variance–covariance matrix for:
- Significance agreement between OD and SD
- Confidence-interval overlap of $\hat{\beta}_{OD}$ vs $\hat{\beta}_{SD}$
- Adjusted $R^2$ degradation

### Handling singular matrices
CART synthesis can produce perfectly collinear columns; singular $X^\top X$ is caught
and recorded as `NaN` rows.


In [6]:
# ── OLS helpers and zero-copy worker ─────────────────────────────────────────

MAX_COEF = 11  # intercept + max 10 slopes (p=10)


def _pad(arr, length=MAX_COEF):
    """Pad/truncate array to fixed length with NaN fill."""
    out = np.full(length, np.nan)
    out[: len(arr)] = arr
    return out


def _fit_ols_arrays(X: np.ndarray, y: np.ndarray):
    """Fit statsmodels OLS on pre-loaded numpy arrays.

    Returns (params, bse, pvalues, adj_rsquared).
    Falls back to NaN arrays on singular / degenerate matrices.
    Imports are local so the function is safely picklable by loky workers.
    """
    import numpy as _np
    import statsmodels.api as _sm

    X_const = _sm.add_constant(X, has_constant="add")
    try:
        res = _sm.OLS(y, X_const).fit()
        return res.params, res.bse, res.pvalues, res.rsquared_adj
    except Exception:
        n = X_const.shape[1]
        nan = _np.full(n, _np.nan)
        return nan, nan, nan, _np.nan


def process_work_item(
    item,
    shm_od_name: str,
    shm_sd_name: str,
    od_shape: tuple,
    sd_shape: tuple,
    max_feat: int,
):
    """Zero-copy worker.

    Attaches to the already-existing SharedMemory arenas by OS name,
    slices this scenario's rows as a *view* (no memcpy), fits OLS,
    then detaches.  Only the tiny result dict is returned over IPC.

    Parameters
    ----------
    item         : (od_start, od_end, sd_start, sd_end, p_val, meta_key)
    shm_od_name  : SharedMemory name of the OD arena
    shm_sd_name  : SharedMemory name of the SD arena
    od_shape     : shape of the OD arena array
    sd_shape     : shape of the SD arena array
    max_feat     : number of feature columns in the arena (== MAX_FEAT)
    """
    import numpy as _np
    from multiprocessing.shared_memory import SharedMemory as _SHM

    od_s, od_e, sd_s, sd_e, p_val, meta_key = item
    method, vtype, N, p_val, rho, sigma, it = meta_key

    # ── Attach to shared memory (no data copy) ────────────────────────────
    # ── Attach to shared memory (no data copy) ────────────────────────────
    # ADD track=False HERE:
    _shm_od = _SHM(name=shm_od_name, create=False, track=False)
    _shm_sd = _SHM(name=shm_sd_name, create=False, track=False)

    arena_od = _np.ndarray(od_shape, dtype=_np.float64, buffer=_shm_od.buf)
    arena_sd = _np.ndarray(sd_shape, dtype=_np.float64, buffer=_shm_sd.buf)

    # ── Slice this scenario (view — still no copy) ────────────────────────
    od_chunk = arena_od[od_s:od_e]   # shape (N, max_feat+1)
    sd_chunk = arena_sd[sd_s:sd_e]

    # Only use the first p_val feature columns; y is always the last column
    X_od = _np.array(od_chunk[:, :p_val],  dtype=_np.float64)   # copy needed by OLS
    y_od = _np.array(od_chunk[:, max_feat], dtype=_np.float64)
    X_sd = _np.array(sd_chunk[:, :p_val],  dtype=_np.float64)
    y_sd = _np.array(sd_chunk[:, max_feat], dtype=_np.float64)

    _shm_od.close()
    _shm_sd.close()

    # ── Fit OLS ───────────────────────────────────────────────────────────
    beta_od, se_od, pval_od, adj_r2_od = _fit_ols_arrays(X_od, y_od)
    beta_sd, se_sd, pval_sd, adj_r2_sd = _fit_ols_arrays(X_sd, y_sd)

    # ── Assemble result row (same schema as original notebook) ────────────
    row = {
        "method":   method,
        "var_type": vtype,
        "N":        N,
        "p":        p_val,
        "rho":      rho,
        "sigma_2":  sigma,
        "iter":     it,
        "adj_r2_od": adj_r2_od,
        "adj_r2_sd": adj_r2_sd,
    }
    for i, v in enumerate(_pad(beta_od)): row[f"beta_od_{i}"] = v
    for i, v in enumerate(_pad(beta_sd)): row[f"beta_sd_{i}"] = v
    for i, v in enumerate(_pad(se_od)):   row[f"se_od_{i}"]   = v
    for i, v in enumerate(_pad(se_sd)):   row[f"se_sd_{i}"]   = v
    for i, v in enumerate(_pad(pval_od)): row[f"pval_od_{i}"] = v
    for i, v in enumerate(_pad(pval_sd)): row[f"pval_sd_{i}"] = v

    return row


In [7]:
# ── Parallel OLS fitting — SharedMemory zero-copy edition ────────────────────
#
# Each worker receives ONLY:
#   • a tiny work_items tuple   (<< 1 KB per item)
#   • the SharedMemory OS names (two short strings)
#   • array shapes and MAX_P (three small ints)
#
# Workers attach to the arenas by name; the OS kernel maps the same physical
# pages into each worker's address space — NO copies, NO per-item pickle.
#
# loky (process-based) backend bypasses the GIL and saturates all cores.

N_JOBS = -1  # use all available cores

print(f"Dispatching {len(work_items):,} work items across {N_JOBS} jobs …")
print(f"OD arena: {OD_SHM_NAME}  SD arena: {SD_SHM_NAME}")
t0 = time.time()

raw_results = Parallel(
    n_jobs=N_JOBS,
    verbose=10,
    batch_size="auto",
    backend="loky",
)(
    delayed(process_work_item)(
        item,
        OD_SHM_NAME, SD_SHM_NAME,
        OD_SHAPE,    SD_SHAPE,
        MAX_P,
    )
    for item in work_items
)

elapsed = time.time() - t0
print(f"\n✓ Finished in {elapsed / 60:.1f} min ({elapsed:.0f} s)")

# ── Release shared memory immediately after workers finish ────────────────────
shm_od.close(); shm_od.unlink()
shm_sd.close(); shm_sd.unlink()
print("Shared memory arenas released.")


Dispatching 216,000 work items across -1 jobs …
OD arena: psm_2de62863  SD arena: psm_f2887e1c


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 24 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:    1.1s
[Parallel(n_jobs=-1)]: Done  13 tasks      | elapsed:    1.1s
[Parallel(n_jobs=-1)]: Done  24 tasks      | elapsed:    1.1s
[Parallel(n_jobs=-1)]: Done  37 tasks      | elapsed:    1.2s
[Parallel(n_jobs=-1)]: Done  50 tasks      | elapsed:    1.2s
[Parallel(n_jobs=-1)]: Batch computation too fast (0.18725052488040778s.) Setting batch_size=2.
[Parallel(n_jobs=-1)]: Done  65 tasks      | elapsed:    1.2s
[Parallel(n_jobs=-1)]: Done  80 tasks      | elapsed:    1.2s
[Parallel(n_jobs=-1)]: Done  97 tasks      | elapsed:    1.2s
[Parallel(n_jobs=-1)]: Done 114 tasks      | elapsed:    1.2s
[Parallel(n_jobs=-1)]: Batch computation too fast (0.009122133255004883s.) Setting batch_size=4.
[Parallel(n_jobs=-1)]: Done 148 tasks      | elapsed:    1.2s
[Parallel(n_jobs=-1)]: Done 184 tasks      | elapsed:    1.3s
[Parallel(n_jobs=-1)]: Batch computation too fas


✓ Finished in 0.2 min (11 s)
Shared memory arenas released.


In [8]:
# ── Assemble master DataFrame ─────────────────────────────────────────────────
# Unmatched OD scenarios were already excluded during work_items construction
# (reported as skipped_no_od). process_work_item always returns a dict, but
# we guard with None-filter for safety.
valid_results = [r for r in raw_results if r is not None]
n_none = len(raw_results) - len(valid_results)

df = pd.DataFrame(valid_results)
del raw_results, valid_results
gc.collect()

print(f"Master DataFrame shape  : {df.shape}")
print(f"Skipped (no matching OD): {skipped_no_od:,}  (filtered during arena build)")
if n_none:
    print(f"Unexpected None results : {n_none:,}")
print(f"Memory usage            : {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
df.head(3)


Master DataFrame shape  : (216000, 75)
Skipped (no matching OD): 0  (filtered during arena build)
Memory usage            : 132.1 MB


,method,var_type,N,p,rho,sigma_2,iter,adj_r2_od,adj_r2_sd,beta_od_0,...,pval_sd_1,pval_sd_2,pval_sd_3,pval_sd_4,pval_sd_5,pval_sd_6,pval_sd_7,pval_sd_8,pval_sd_9,pval_sd_10
0,cart,binary,100,3,0.0,1.0,1,0.451085,0.510875,1.047472,...,7.143577e-16,1.108712e-11,5.158796e-14,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,cart,binary,100,3,0.0,1.0,2,0.384715,0.409975,0.887732,...,2.799741e-08,1.329822e-16,1.098782e-11,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,cart,binary,100,3,0.0,1.0,3,0.435418,0.349908,0.982210,...,1.797476e-11,2.856083e-14,5.093723e-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
# ── Save to Parquet ─────────────────────────────────────────────────────────
parquet_path = os.path.join(result_dir, "aggregated_model_metrics.parquet")
df.to_parquet(parquet_path, index=False, engine="pyarrow")

file_size_mb = os.path.getsize(parquet_path) / 1e6
print(f"✓ Saved to: {parquet_path}")
print(f"  File size: {file_size_mb:.1f} MB")
print(f"  Rows: {len(df):,}  •  Columns: {len(df.columns)}")

✓ Saved to: ../results/aggregated_model_metrics.parquet
  File size: 72.7 MB
  Rows: 216,000  •  Columns: 75


## Quick Sanity Checks

In [10]:
# ── Check for NaN rows (failed fits) ────────────────────────────────────────
nan_rows = df["adj_r2_od"].isna().sum()
nan_sd   = df["adj_r2_sd"].isna().sum()
print(f"Failed OD fits (NaN adj_r2_od): {nan_rows:,}")
print(f"Failed SD fits (NaN adj_r2_sd): {nan_sd:,}")

if nan_sd > 0:
    print("\nBreakdown of failed SD fits by method:")
    print(df[df["adj_r2_sd"].isna()].groupby("method").size())

Failed OD fits (NaN adj_r2_od): 0
Failed SD fits (NaN adj_r2_sd): 0


In [11]:
# ── Summary: mean Adjusted R² by method and var_type ────────────────────────
summary = (
    df.dropna(subset=["adj_r2_od", "adj_r2_sd"])
      .groupby(["method", "var_type"])
      .agg(
          n_pairs     = ("adj_r2_od", "count"),
          adj_r2_od   = ("adj_r2_od", "mean"),
          adj_r2_sd   = ("adj_r2_sd", "mean"),
          r2_gap_mean = ("adj_r2_od", lambda x: (x - df.loc[x.index, "adj_r2_sd"]).mean()),
      )
      .round(4)
)
summary

n_pairs  adj_r2_od  adj_r2_sd  r2_gap_mean
method var_type                                              
cart   binary        36000     0.6218     0.5712       0.0506
       continuous    36000     0.8428     0.6336       0.2092
norm   binary        36000     0.6218     0.6148       0.0070
       continuous    36000     0.8428     0.5919       0.2508
pmm    binary        36000     0.6218     0.6275      -0.0057
       continuous    36000     0.8428     0.8343       0.0085

In [12]:
# ── Verify coefficient columns look reasonable ──────────────────────────────
# For p=3 scenarios, columns beta_*_4 through beta_*_10 should be NaN.
p3_mask = df["p"] == 3
print(f"p=3 rows: {p3_mask.sum():,}")
print(f"  beta_od_4 all NaN? {df.loc[p3_mask, 'beta_od_4'].isna().all()}")
print(f"  beta_od_3 has values? {df.loc[p3_mask, 'beta_od_3'].notna().any()}")

p10_mask = df["p"] == 10
print(f"\np=10 rows: {p10_mask.sum():,}")
print(f"  beta_od_10 has values? {df.loc[p10_mask, 'beta_od_10'].notna().any()}")

print("\n✓ Notebook 03 complete. Proceed to 04_evaluation.ipynb.")

p=3 rows: 108,000
  beta_od_4 all NaN? True
  beta_od_3 has values? True

p=10 rows: 108,000
  beta_od_10 has values? True

✓ Notebook 03 complete. Proceed to 04_evaluation.ipynb.
